In [24]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings('ignore')


# ── Robust chart export + inline display ──────────────────────────────────────
def save_chart(fig, path_no_ext, width=1100, height=700):
    """Display inline in notebook, then save .html and attempt .png."""
    fig.show()

    fig.write_html(f"{path_no_ext}.html", config=WRITE_CONFIG)
    print(f"  ✓ {path_no_ext}.html")

    for engine in ["kaleido", "orca"]:
        try:
            fig.write_image(f"{path_no_ext}.png", width=width, height=height,
                            scale=3, engine=engine)
            print(f"  ✓ {path_no_ext}.png ({engine})")
            return
        except Exception:
            pass

    print(f"  ⚠ PNG export skipped (kaleido not available). HTML saved.")

# ── Style system (matching user's existing charts) ────────────────────────────
STYLE = {
    'font_family': 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size': 10,
    'axis_title_size': 14,
    'legend_size': 10,
    'annotation_size': 13,
    'title_color': '#1a2744',
    'template': 'plotly_white',
    'plot_bg': '#ffffff',
    'paper_bg': '#ffffff',
    'chart_height': 550,
    'chart_height_small': 420,
    'chart_height_tall': 700,
    'margin': dict(l=60, r=40, t=10, b=50),
    'margin_bar': dict(l=160, r=130, t=10, b=50),
    'grid_color': '#e5e7eb',
    'grid_width': 0.5,
    'zero_line_color': '#c9cfd6',
}
WRITE_CONFIG = {'displayModeBar': False, 'responsive': True}

# Palette drawn from the user's existing colour system
PALETTE = {
    'blue':       '#4a6fa5',
    'red':        '#c23a3a',
    'green':      '#2e7d4a',
    'orange':     '#d4853b',
    'light_blue': '#7a9dc4',
    'light_red':  '#d46b6b',
    'light_green':'#5aa87a',
    'dark':       '#3d4f5f',
    'grey':       '#999999',
    'gold':       '#e6b980',
}

def base_layout(**kwargs):
    layout = dict(
        template=STYLE['template'],
        plot_bgcolor=STYLE['plot_bg'],
        paper_bgcolor=STYLE['paper_bg'],
        font=dict(family=STYLE['font_family'], size=STYLE['tick_size'],
                  color=STYLE['title_color']),
        margin=STYLE['margin'],
        height=STYLE['chart_height'],
    )
    layout.update(kwargs)
    return layout

# ── Output directory ──────────────────────────────────────────────────────────
OUT = os.path.join('..', 'figures', 'ml')
os.makedirs(OUT, exist_ok=True)

# ── 0. Load & prepare data ───────────────────────────────────────────────────
filter_flag = True

GITHUB_BASE = "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/main/MASTER/"
master = pd.read_csv(f"{GITHUB_BASE}Master.csv").drop(columns=['Unnamed: 0'], errors='ignore')
print(f"Master: {len(master)} obs, {master['Country Code'].nunique()} countries")

df = master[(master['Year'] >= 1995) & (master['Year'] <= 2019)].dropna(subset=['Economic Complexity Index']).copy()
df = df.sort_values(['Country Code', 'Year'])

resource_rich_countries = master[master['Year'] == 1995][
    master[master['Year'] == 1995]['Total natural resources rents (% of GDP)'] >= 5
]['Country Code'].unique()

if filter_flag:
    df = df[df['Country Code'].isin(resource_rich_countries)]
df['Resource_Rich'] = df['Country Code'].isin(resource_rich_countries).astype(int)

df['L1_ECI'] = df.groupby('Country Code')['Economic Complexity Index'].shift(1)
df = df.dropna()

df['HCI_x_ProductionValue']       = df['Human capital index'] * df['Total_Production_Value']
df['RuleOfLaw_x_ProductionValue'] = df['Rule of law index']   * df['Total_Production_Value']

base_features = [
    'Total_Production_Value',
    'Human capital index',
    'Rule of law index',
    'Political stability — estimate',
    'Trade (% of GDP)',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Share of investment in GDP',
    'Domestic credit to private sector (% of GDP)',
    'Landlocked',
    'Urban population (% of total population)',
    'Government revenue',
    'Capital depreciation rate',
    'Use of IMF credit (DOD, current US$)',
    'Real interest rate (%)',
    'Inflation, consumer prices (annual %)',
    'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)',
    'L1_ECI'
]
all_features = base_features + ['HCI_x_ProductionValue', 'RuleOfLaw_x_ProductionValue']
print(f"Total features: {len(all_features)} (18 base + 1 group + 2 interactions)")

# ── 1. Log transforms ────────────────────────────────────────────────────────
min_val = df['Economic Complexity Index'].min()
df['ECI_shifted'] = df['Economic Complexity Index'] - min_val + 1
df['Economic Complexity Index'] = np.log(df['ECI_shifted'])

min_val = df['L1_ECI'].min()
df['L1_ECI_shifted'] = df['L1_ECI'] - min_val + 1
df['L1_ECI'] = np.log(df['L1_ECI_shifted'])

log_features = [
    'Human capital index',
    'Total_Production_Value',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Government revenue',
    'Use of IMF credit (DOD, current US$)',
]
df[log_features] = np.log1p(df[log_features])
df[log_features] = df[log_features].replace([np.inf, -np.inf], np.nan)
df = df.dropna()
print(f"After transforms: {len(df)} obs")

# ── 2. Arrays ─────────────────────────────────────────────────────────────────
y = df['Economic Complexity Index'].values
X = df[all_features].copy()
print(f"Observations: {X.shape[0]} | Features: {X.shape[1]}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(X.values)
y_train = y

tscv = TimeSeriesSplit(n_splits=5, gap=1)

# ── 3. Train models ──────────────────────────────────────────────────────────
lasso   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_train)
ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_train)
elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_train)
rf      = RandomForestRegressor(
    n_estimators=200, max_depth=4, min_samples_leaf=10,
    random_state=42, n_jobs=-1, oob_score=True
).fit(X_train, y_train)

models = {'LASSO': lasso, 'Ridge': ridge, 'Elastic Net': elastic, 'Random Forest': rf}
print("\n✓ All models fitted")
print(f"  Elastic Net  →  l1_ratio: {elastic.l1_ratio_:.2f}  |  alpha: {elastic.alpha_:.4f}")
print(f"  LASSO        →  alpha:    {lasso.alpha_:.4f}")
print(f"  Ridge        →  alpha:    {ridge.alpha_:.4f}")
print(f"  RF OOB R²    →  {rf.oob_score_:.4f}")

# ── 4. Performance table ─────────────────────────────────────────────────────
perf_rows = []
for name, model in models.items():
    pred = model.predict(X_train)
    perf_rows.append({
        'Model': name,
        'R²': round(model.score(X_train, y_train), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_train, pred)), 4),
        'MAE': round(mean_absolute_error(y_train, pred), 4),
    })
perf_df = pd.DataFrame(perf_rows).sort_values('R²', ascending=False).reset_index(drop=True)
print("\n", perf_df.to_string(index=False))

# ── 5. VIF ────────────────────────────────────────────────────────────────────
vif_data = pd.DataFrame({
    'Feature': all_features,
    'VIF': [variance_inflation_factor(X_train, i) for i in range(X_train.shape[1])]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

# ── 6. Importance ─────────────────────────────────────────────────────────────
def minmax(arr):
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo) if hi > lo else arr

all_importance = pd.DataFrame({'Feature': all_features})
for name, model in models.items():
    if name == 'Random Forest':
        all_importance[name] = minmax(model.feature_importances_)
    else:
        all_importance[name] = minmax(np.abs(model.coef_))

# ── Name mapping ──────────────────────────────────────────────────────────────
name_mapping = {
    'Human capital index':                                                   'Human Capital',
    'Rule of law index':                                                     'Rule of Law',
    'Political stability — estimate':                                        'Political Stability',
    'Trade (% of GDP)':                                                      'Trade',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP':   'Capital Formation',
    'Share of investment in GDP':                                            'Investment Share',
    'Domestic credit to private sector (% of GDP)':                         'Domestic Credit',
    'Landlocked':                                                            'Landlocked',
    'Urban population (% of total population)':                             'Urban Population',
    'Government revenue':                                                    'Gov Revenue',
    'Capital depreciation rate':                                             'Depreciation',
    'Use of IMF credit (DOD, current US$)':                                 'IMF Credit',
    'Real interest rate (%)':                                                'Interest Rate',
    'Inflation, consumer prices (annual %)':                                 'Inflation',
    'Access to electricity (% of population)':                              'Electricity',
    'Adjusted savings: gross savings (% of GNI)':                           'Savings',
    'Total_Production_Value':                                                'Production Value',
    'HCI_x_ProductionValue':                                                 'HC × Production',
    'RuleOfLaw_x_ProductionValue':                                           'RuleLaw × Production',
    'L1_ECI':                                                                'Lagged ECI',
}
def shorten(name):
    return name_mapping.get(name, name[:22])

EXCLUDE = 'L1_ECI'

# ══════════════════════════════════════════════════════════════════════════════
# CHART 1: VIF
# ══════════════════════════════════════════════════════════════════════════════
print("\n[1/4] VIF chart...")

vif_plot = vif_data.copy()
vif_plot['Short'] = vif_plot['Feature'].apply(shorten)
vif_plot = vif_plot.sort_values('VIF', ascending=True).reset_index(drop=True)

colors_vif = [PALETTE['red'] if v > 10 else PALETTE['blue'] for v in vif_plot['VIF']]

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    y=vif_plot['Short'],
    x=vif_plot['VIF'],
    orientation='h',
    marker=dict(color=colors_vif, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.1f}' for v in vif_plot['VIF']],
    textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
))

# Threshold lines
fig1.add_vline(x=10, line=dict(color=PALETTE['red'], width=2, dash='dash'),
               annotation_text='VIF = 10', annotation_position='right',
               annotation_font=dict(size=STYLE['annotation_size'], color=PALETTE['red']))
fig1.add_vline(x=5, line=dict(color=PALETTE['orange'], width=1.5, dash='dot'),
               annotation_text='VIF = 5', annotation_position='right',
               annotation_font=dict(size=STYLE['annotation_size'], color=PALETTE['orange']))

fig1.update_layout(**base_layout(
    height=STYLE['chart_height_tall'],
    margin=STYLE['margin_bar'],
    xaxis=dict(
        title=dict(text='Variance Inflation Factor (VIF)', font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        range=[0, max(vif_plot['VIF'].max() * 1.15, 12)],
    ),
    yaxis=dict(
        tickfont=dict(size=STYLE['tick_size']),
    ),
    showlegend=False,
))

save_chart(fig1, os.path.join(OUT, 'VIF_model2_resource_rich'), width=1100, height=700)

# ══════════════════════════════════════════════════════════════════════════════
# CHART 2: 3-Panel Coefficient Comparison
# ══════════════════════════════════════════════════════════════════════════════
print("\n[2/4] 3-panel coefficient comparison...")

model_names_lin = ['LASSO', 'Ridge', 'Elastic Net']

fig2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"<b>{n}</b><br><span style='font-size:11px;color:#666'>R² = {perf_df[perf_df['Model']==n]['R²'].values[0]:.3f} | "
        f"{int(np.sum(models[n].coef_ != 0))} features</span>"
        for n in model_names_lin
    ],
    horizontal_spacing=0.08,
)

for col_idx, model_name in enumerate(model_names_lin, 1):
    signed_vals = models[model_name].coef_.copy()
    abs_vals    = np.abs(signed_vals)

    exclude_idx = all_features.index(EXCLUDE) if EXCLUDE in all_features else None
    if exclude_idx is not None:
        abs_vals[exclude_idx] = -np.inf

    top15_idx  = np.argsort(abs_vals)[::-1][:15]
    top15_feat = [all_features[i] for i in top15_idx]
    top15_vals = signed_vals[top15_idx]

    colors = [PALETTE['green'] if v > 0 else PALETTE['red'] for v in top15_vals]
    labels = [shorten(f) for f in top15_feat]

    # Reverse for plotly (bottom-to-top)
    labels_r = labels[::-1]
    vals_r   = top15_vals[::-1]
    colors_r = colors[::-1]

    fig2.add_trace(go.Bar(
        y=labels_r,
        x=vals_r,
        orientation='h',
        marker=dict(color=colors_r, line=dict(color='#1a2744', width=0.5)),
        text=[f'{v:+.3f}' if abs(v) > 0.005 else '' for v in vals_r],
        textposition='outside',
        textfont=dict(size=9, color=STYLE['title_color']),
        showlegend=False,
    ), row=1, col=col_idx)

    # Zero line
    fig2.add_vline(x=0, line=dict(color=STYLE['zero_line_color'], width=2),
                   row=1, col=col_idx)

    fig2.update_xaxes(
        title_text='Coefficient' if col_idx == 2 else '',
        title_font=dict(size=STYLE['axis_title_size']),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        row=1, col=col_idx,
    )
    fig2.update_yaxes(
        tickfont=dict(size=STYLE['tick_size']),
        row=1, col=col_idx,
    )

# Legend via invisible traces
fig2.add_trace(go.Bar(y=[None], x=[None], marker_color=PALETTE['green'],
                       name='Positive', showlegend=True))
fig2.add_trace(go.Bar(y=[None], x=[None], marker_color=PALETTE['red'],
                       name='Negative', showlegend=True))

fig2.update_layout(**base_layout(
    height=STYLE['chart_height_tall'],
    width=1600,
    margin=dict(l=160, r=80, t=70, b=100),
    legend=dict(
        orientation='h', yanchor='top', y=-0.15, xanchor='center', x=0.5,
        font=dict(size=STYLE['legend_size']),
    ),
))

save_chart(fig2, os.path.join(OUT, 'Coef_Comparison_model2_resource_rich'), width=1600, height=700)

# ══════════════════════════════════════════════════════════════════════════════
# CHART 3: Model Agreement — Dot-and-Range
# ══════════════════════════════════════════════════════════════════════════════
print("\n[3/4] Model agreement chart...")

imp_df = all_importance[all_importance['Feature'] != EXCLUDE].copy()
imp_df['EN_abs'] = imp_df['Elastic Net']
top12  = imp_df.sort_values('EN_abs', ascending=False).head(12).reset_index(drop=True)
top12['Short'] = top12['Feature'].apply(shorten)

# Reverse for plotly (bottom-to-top)
top12_r = top12.iloc[::-1].reset_index(drop=True)

fig3 = go.Figure()

# Range lines
for _, row in top12_r.iterrows():
    vals = [row['LASSO'], row['Ridge'], row['Elastic Net']]
    fig3.add_trace(go.Scatter(
        x=[min(vals), max(vals)],
        y=[row['Short'], row['Short']],
        mode='lines',
        line=dict(color='#aab0b8', width=3),
        showlegend=False,
        hoverinfo='skip',
    ))

# Model dots
dot_config = [
    ('LASSO',       'circle',       PALETTE['red']),
    ('Ridge',       'square',       PALETTE['blue']),
    ('Elastic Net', 'triangle-up',  PALETTE['green']),
]
for model_name, symbol, color in dot_config:
    fig3.add_trace(go.Scatter(
        x=top12_r[model_name],
        y=top12_r['Short'],
        mode='markers',
        marker=dict(symbol=symbol, size=12, color=color,
                    line=dict(color='#1a2744', width=1)),
        name=model_name,
        hovertemplate='%{y}: %{x:.3f}<extra>' + model_name + '</extra>',
    ))

fig3.update_layout(**base_layout(
    height=STYLE['chart_height'],
    margin=STYLE['margin_bar'],
    xaxis=dict(
        title=dict(text='Normalised Feature Importance',
                   font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        range=[-0.02, top12[['LASSO', 'Ridge', 'Elastic Net']].max().max() + 0.08],
    ),
    yaxis=dict(tickfont=dict(size=STYLE['tick_size'])),
    legend=dict(
        font=dict(size=STYLE['legend_size']),
        yanchor='bottom', y=0.02, xanchor='right', x=0.98,
        bgcolor='rgba(255,255,255,0.95)', bordercolor='#e5e7eb', borderwidth=1,
    ),
))

save_chart(fig3, os.path.join(OUT, 'ModelAgreement_model2_resource_rich'), width=1100, height=600)

# ══════════════════════════════════════════════════════════════════════════════
# CHART 4: Random Forest Feature Importance
# ══════════════════════════════════════════════════════════════════════════════
print("\n[4/4] Random Forest importance...")

rf_imp = pd.DataFrame({'Feature': all_features, 'Importance': rf.feature_importances_})
rf_top = (
    rf_imp[rf_imp['Feature'] != EXCLUDE]
    .sort_values('Importance', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

# Colour gradient from light to dark using the orange/gold palette
norm_vals = rf_top['Importance'] / rf_top['Importance'].max()
def lerp_color(t, c_low=(230, 185, 128), c_high=(180, 80, 40)):
    return f'rgb({int(c_low[0]+(c_high[0]-c_low[0])*t)},{int(c_low[1]+(c_high[1]-c_low[1])*t)},{int(c_low[2]+(c_high[2]-c_low[2])*t)})'

bar_colors_rf = [lerp_color(v) for v in norm_vals]

# Reverse for plotly
rf_top_r = rf_top.iloc[::-1].reset_index(drop=True)
bar_colors_rf_r = bar_colors_rf[::-1]

fig4 = go.Figure()

fig4.add_trace(go.Bar(
    y=[shorten(f) for f in rf_top_r['Feature']],
    x=rf_top_r['Importance'],
    orientation='h',
    marker=dict(color=bar_colors_rf_r, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.4f}' for v in rf_top_r['Importance']],
    textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
))

fig4.update_layout(**base_layout(
    height=STYLE['chart_height_tall'],
    margin=STYLE['margin_bar'],
    xaxis=dict(
        title=dict(text='Mean Decrease in Impurity',
                   font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        range=[0, rf_top['Importance'].max() * 1.22],
    ),
    yaxis=dict(tickfont=dict(size=STYLE['tick_size'])),
    showlegend=False,
    annotations=[
        dict(
            text=f"OOB R² = {rf.oob_score_:.3f}  |  200 trees  |  max_depth = 4",
            xref='paper', yref='paper', x=0.98, y=0.02,
            showarrow=False,
            font=dict(size=STYLE['annotation_size'], color='#666'),
            bgcolor='rgba(255,255,255,0.95)', bordercolor='#e5e7eb', borderwidth=1,
            borderpad=6,
        )
    ],
))

save_chart(fig4, os.path.join(OUT, 'RF_model2_resource_rich'), width=1100, height=700)

print(f"\n✓ All 4 charts saved to {OUT}/")
print("  Formats: .html (interactive), .png (3× retina), .pdf (vector)")


Master: 3150 obs, 126 countries
Total features: 20 (18 base + 1 group + 2 interactions)
After transforms: 1344 obs
Observations: 1344 | Features: 20

✓ All models fitted
  Elastic Net  →  l1_ratio: 0.50  |  alpha: 0.0096
  LASSO        →  alpha:    0.0059
  Ridge        →  alpha:    15.1991
  RF OOB R²    →  0.7365

         Model     R²   RMSE    MAE
Random Forest 0.7862 0.1059 0.0659
        Ridge 0.7481 0.1149 0.0753
  Elastic Net 0.7408 0.1166 0.0746
        LASSO 0.7373 0.1174 0.0748

[1/4] VIF chart...


  ✓ ..\figures\ml\VIF_model2_resource_rich.html
  ✓ ..\figures\ml\VIF_model2_resource_rich.png (kaleido)

[2/4] 3-panel coefficient comparison...


  ✓ ..\figures\ml\Coef_Comparison_model2_resource_rich.html
  ✓ ..\figures\ml\Coef_Comparison_model2_resource_rich.png (kaleido)

[3/4] Model agreement chart...


  ✓ ..\figures\ml\ModelAgreement_model2_resource_rich.html
  ✓ ..\figures\ml\ModelAgreement_model2_resource_rich.png (kaleido)

[4/4] Random Forest importance...


  ✓ ..\figures\ml\RF_model2_resource_rich.html
  ✓ ..\figures\ml\RF_model2_resource_rich.png (kaleido)

✓ All 4 charts saved to ..\figures\ml/
  Formats: .html (interactive), .png (3× retina), .pdf (vector)


# Robustness checks

## Bootstrap

In [21]:
# ── Bootstrap Stability (100 runs) — LASSO, Ridge and Elastic Net ─────────────
print("\n── Bootstrap Feature Stability (100 runs) ──────────────────────────────")

rng    = np.random.default_rng(42)
n_boot = 100

lasso_counts = np.zeros(len(all_features))
en_counts    = np.zeros(len(all_features))
ridge_coefs  = np.zeros((n_boot, len(all_features)))

for b in range(n_boot):
    idx    = rng.choice(len(X_train), size=len(X_train), replace=True)
    Xb, yb = X_train[idx], y_train[idx]

    bl  = LassoCV(cv=3, random_state=None, max_iter=10000).fit(Xb, yb)
    lasso_counts += (bl.coef_ != 0).astype(int)

    ben = ElasticNetCV(l1_ratio=0.5, cv=3, random_state=None, max_iter=10000).fit(Xb, yb)
    en_counts += (ben.coef_ != 0).astype(int)

    br  = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=3).fit(Xb, yb)
    ridge_coefs[b] = br.coef_

ridge_coef_mean = np.mean(ridge_coefs, axis=0)
ridge_coef_std  = np.std(ridge_coefs,  axis=0)

stability_boot = pd.DataFrame({
    'Feature':        all_features,
    'LASSO_Rate':     (lasso_counts / n_boot).round(3),
    'EN_Rate':        (en_counts    / n_boot).round(3),
    'Ridge_CoefMean': ridge_coef_mean.round(4),
    'Ridge_CoefStd':  ridge_coef_std.round(4),
})
stability_boot = (
    stability_boot[stability_boot['Feature'] != EXCLUDE]
    .copy()
    .reset_index(drop=True)
    .sort_values('EN_Rate', ascending=False)
    .reset_index(drop=True)
)
stability_boot['Short'] = stability_boot['Feature'].apply(shorten)

# Summary counts
for name, col in [('LASSO', 'LASSO_Rate'), ('Elastic Net', 'EN_Rate')]:
    n_stable   = (stability_boot[col] >= 0.70).sum()
    n_unstable = (stability_boot[col] <  0.30).sum()
    print(f"  {name:<12} — Stable (≥70%): {n_stable}  |  Unstable (<30%): {n_unstable}")
print(stability_boot[['Feature', 'LASSO_Rate', 'EN_Rate',
                       'Ridge_CoefMean', 'Ridge_CoefStd']].to_string(index=False))

# ── CHART: Bootstrap Stability ────────────────────────────────────────────────
print("\n── Plotting bootstrap stability chart...")

# Sort ascending for plotly (bottom = lowest)
plot_df = stability_boot.sort_values('EN_Rate', ascending=True).reset_index(drop=True)

def sel_color(val):
    if val >= 0.70: return PALETTE['green']
    elif val >= 0.30: return PALETTE['orange']
    else: return PALETTE['red']

lasso_colors = [sel_color(v) for v in plot_df['LASSO_Rate']]
en_colors    = [sel_color(v) for v in plot_df['EN_Rate']]

# Ridge: normalise std for gradient
std_vals     = plot_df['Ridge_CoefStd'].values
norm_std     = (std_vals - std_vals.min()) / (std_vals.max() - std_vals.min() + 1e-8)
ridge_colors = [lerp_color(v, c_low=(180, 220, 180), c_high=(200, 60, 60))
                for v in norm_std]

fig_boot = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>LASSO</b><br><span style="font-size:11px;color:#666">Selection Rate across 100 Bootstrap Samples</span>',
        '<b>Elastic Net</b><br><span style="font-size:11px;color:#666">Selection Rate across 100 Bootstrap Samples</span>',
        '<b>Ridge</b><br><span style="font-size:11px;color:#666">Coefficient Std Dev across 100 Bootstrap Samples</span>',
    ],
    horizontal_spacing=0.10,
)

# ── Panel 1: LASSO ────────────────────────────────────────────────────────────
fig_boot.add_trace(go.Bar(
    y=plot_df['Short'],
    x=plot_df['LASSO_Rate'],
    orientation='h',
    marker=dict(color=lasso_colors, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.2f}' for v in plot_df['LASSO_Rate']],
    textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
    showlegend=False,
    hovertemplate='<b>%{y}</b><br>Selection Rate: %{x:.3f}<extra>LASSO</extra>',
), row=1, col=1)

for xval, color, dash in [
    (0.70, PALETTE['green'], 'dash'),
    (0.30, PALETTE['red'],   'dot'),
]:
    fig_boot.add_vline(
        x=xval, line=dict(color=color, width=1.5, dash=dash),
        row=1, col=1,
    )

# ── Panel 2: Elastic Net ──────────────────────────────────────────────────────
fig_boot.add_trace(go.Bar(
    y=plot_df['Short'],
    x=plot_df['EN_Rate'],
    orientation='h',
    marker=dict(color=en_colors, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.2f}' for v in plot_df['EN_Rate']],
    textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
    showlegend=False,
    hovertemplate='<b>%{y}</b><br>Selection Rate: %{x:.3f}<extra>Elastic Net</extra>',
), row=1, col=2)

for xval, color, dash in [
    (0.70, PALETTE['green'], 'dash'),
    (0.30, PALETTE['red'],   'dot'),
]:
    fig_boot.add_vline(
        x=xval, line=dict(color=color, width=1.5, dash=dash),
        row=1, col=2,
    )

# ── Panel 3: Ridge ────────────────────────────────────────────────────────────
fig_boot.add_trace(go.Bar(
    y=plot_df['Short'],
    x=plot_df['Ridge_CoefStd'],
    orientation='h',
    marker=dict(color=ridge_colors, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.4f}' for v in plot_df['Ridge_CoefStd']],
    textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
    showlegend=False,
    hovertemplate='<b>%{y}</b><br>Coef Std Dev: %{x:.4f}<extra>Ridge</extra>',
), row=1, col=3)

# ── Shared dummy legend for colour categories ─────────────────────────────────
for label, color in [
    ('Stable (≥70%)',      PALETTE['green']),
    ('Moderate (30–70%)',  PALETTE['orange']),
    ('Unstable (<30%)',    PALETTE['red']),
]:
    fig_boot.add_trace(go.Bar(
        y=[None], x=[None], orientation='h',
        marker=dict(color=color, line=dict(color='#1a2744', width=0.5)),
        name=label, showlegend=True,
    ))

# ── Threshold annotations below x-axis (panels 1 & 2) ────────────────────────
for xref, col_ref in [('x', 'panel 1'), ('x2', 'panel 2')]:
    fig_boot.add_annotation(
        x=0.70, y=-0.10, xref=xref, yref='paper',
        text='Stable<br>(70%)', showarrow=False,
        font=dict(size=9, color=PALETTE['green']),
        xanchor='center', yanchor='top',
    )
    fig_boot.add_annotation(
        x=0.30, y=-0.10, xref=xref, yref='paper',
        text='Unstable<br>(30%)', showarrow=False,
        font=dict(size=9, color=PALETTE['red']),
        xanchor='center', yanchor='top',
    )

# ── Axes styling ──────────────────────────────────────────────────────────────
for col in [1, 2]:
    fig_boot.update_xaxes(
        range=[0, 1.20],
        title_text='Selection Rate' if col == 1 else '',
        title_font=dict(size=STYLE['axis_title_size']),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        row=1, col=col,
    )

fig_boot.update_xaxes(
    title_text='Coefficient Std Dev',
    title_font=dict(size=STYLE['axis_title_size']),
    gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
    range=[0, plot_df['Ridge_CoefStd'].max() * 1.22],
    row=1, col=3,
)

for col in [1, 2, 3]:
    fig_boot.update_yaxes(
        tickfont=dict(size=STYLE['tick_size']),
        showticklabels=(col == 1),
        row=1, col=col,
    )

fig_boot.update_layout(**base_layout(
    height=STYLE['chart_height_tall'],
    width=1600,
    margin=dict(l=180, r=80, t=80, b=130),
    legend=dict(
        orientation='h', yanchor='top', y=-0.18,
        xanchor='center', x=0.38,
        font=dict(size=STYLE['legend_size']),
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#e5e7eb', borderwidth=1,
    ),
))

save_chart(fig_boot,
           os.path.join(OUT, 'Bootstrap_Stability_model2_resource_rich'),
           width=1600, height=700)
print("  ✓ Saved Bootstrap_Stability_model2_resource_rich.{html,png}")


── Bootstrap Feature Stability (100 runs) ──────────────────────────────
  LASSO        — Stable (≥70%): 18  |  Unstable (<30%): 0
  Elastic Net  — Stable (≥70%): 17  |  Unstable (<30%): 0
                                                            Feature  LASSO_Rate  EN_Rate  Ridge_CoefMean  Ridge_CoefStd
                       Domestic credit to private sector (% of GDP)        1.00     1.00          0.0287         0.0052
                                              HCI_x_ProductionValue        1.00     1.00          0.0193         0.0052
                                          Capital depreciation rate        0.99     0.99         -0.0101         0.0035
                            Access to electricity (% of population)        0.98     0.98          0.0293         0.0072
                                     Political stability — estimate        0.97     0.97          0.0131         0.0055
                                             Total_Production_Value        0.96     0.96  

  ✓ ..\figures\ml\Bootstrap_Stability_model2_resource_rich.html
  ✓ ..\figures\ml\Bootstrap_Stability_model2_resource_rich.png (kaleido)
  ✓ Saved Bootstrap_Stability_model2_resource_rich.{html,png}


## Leave-One-Country-Out (LOCO)

In [15]:
# ── LOCO: Leave-One-Country-Out Robustness Check ──────────────────────────────
print("\n── LOCO Robustness Check ───────────────────────────────────────────────")

countries    = df['Country Code'].unique()
loco_coefs   = []
loco_r2      = []
loco_countries = []

for country in countries:
    # Drop all observations for this country
    mask   = df['Country Code'] != country
    X_loco = scaler.fit_transform(X[mask].values)
    y_loco = y[mask]

    if len(X_loco) < 30:          # skip if too few obs
        continue

    tscv_loco = TimeSeriesSplit(n_splits=3, gap=1)
    en_loco   = ElasticNetCV(
        l1_ratio=0.5, cv=tscv_loco,
        random_state=42, max_iter=10000
    ).fit(X_loco, y_loco)

    loco_coefs.append(en_loco.coef_)
    loco_r2.append(en_loco.score(X_loco, y_loco))
    loco_countries.append(country)

loco_coefs = np.array(loco_coefs)          # shape: (n_countries, n_features)
print(f"  LOCO runs completed: {len(loco_countries)} countries")

# ── Summary stats across LOCO runs ───────────────────────────────────────────
loco_mean  = loco_coefs.mean(axis=0)
loco_std   = loco_coefs.std(axis=0)
loco_min   = loco_coefs.min(axis=0)
loco_max   = loco_coefs.max(axis=0)

# Sign consistency: proportion of runs where sign matches full-model sign
full_coef  = elastic.coef_
sign_consistency = np.mean(np.sign(loco_coefs) == np.sign(full_coef), axis=0)

loco_df = pd.DataFrame({
    'Feature':       all_features,
    'Full_Coef':     full_coef.round(4),
    'LOCO_Mean':     loco_mean.round(4),
    'LOCO_Std':      loco_std.round(4),
    'LOCO_Min':      loco_min.round(4),
    'LOCO_Max':      loco_max.round(4),
    'Sign_Consist':  sign_consistency.round(3),
})
loco_df = (
    loco_df[loco_df['Feature'] != EXCLUDE]
    .copy()
    .reset_index(drop=True)
)
loco_df['Short']    = loco_df['Feature'].apply(shorten)
loco_df['abs_coef'] = loco_df['Full_Coef'].abs()
loco_df = loco_df.sort_values('abs_coef', ascending=False).reset_index(drop=True)

print("\n── LOCO Summary (sorted by |Full Coef|) ────────────────────────────────")
print(loco_df[['Short', 'Full_Coef', 'LOCO_Mean', 'LOCO_Std',
               'LOCO_Min', 'LOCO_Max', 'Sign_Consist']].to_string(index=False))

# ── R² stability across LOCO runs ────────────────────────────────────────────
print(f"\n  Full-model R²:  {elastic.score(X_train, y_train):.4f}")
print(f"  LOCO R² mean:   {np.mean(loco_r2):.4f}")
print(f"  LOCO R² std:    {np.std(loco_r2):.4f}")
print(f"  LOCO R² range:  [{min(loco_r2):.4f}, {max(loco_r2):.4f}]")

# ── CHART 1: Coefficient stability — full coef + LOCO range ──────────────────
print("\n── Plotting LOCO charts...")

# Keep top 12 by |full coef|, reverse for plotly
plot_loco = loco_df.head(12).iloc[::-1].reset_index(drop=True)

fig_loco1 = go.Figure()

# LOCO range (min–max band as error bar)
fig_loco1.add_trace(go.Scatter(
    x=plot_loco['LOCO_Mean'],
    y=plot_loco['Short'],
    mode='markers',
    marker=dict(
        symbol='circle', size=10,
        color=PALETTE['blue'],
        line=dict(color='#1a2744', width=1),
    ),
    error_x=dict(
        type='data',
        symmetric=False,
        array=(plot_loco['LOCO_Max'] - plot_loco['LOCO_Mean']).values,
        arrayminus=(plot_loco['LOCO_Mean'] - plot_loco['LOCO_Min']).values,
        color='#aab0b8',
        thickness=2,
        width=6,
    ),
    name='LOCO Mean ± Range',
    hovertemplate='<b>%{y}</b><br>LOCO Mean: %{x:.4f}<extra></extra>',
))

# Full model coefficient
fig_loco1.add_trace(go.Scatter(
    x=plot_loco['Full_Coef'],
    y=plot_loco['Short'],
    mode='markers',
    marker=dict(
        symbol='diamond', size=11,
        color=PALETTE['orange'],
        line=dict(color='#1a2744', width=1),
    ),
    name='Full-sample Coef',
    hovertemplate='<b>%{y}</b><br>Full Coef: %{x:.4f}<extra></extra>',
))

# Zero line
fig_loco1.add_vline(
    x=0, line=dict(color=STYLE['zero_line_color'], width=2, dash='dash'),
)

fig_loco1.update_layout(**base_layout(
    height=STYLE['chart_height'],
    margin=STYLE['margin_bar'],
    xaxis=dict(
        title=dict(text='Standardised Coefficient',
                   font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
    ),
    yaxis=dict(tickfont=dict(size=STYLE['tick_size'])),
    legend=dict(
        font=dict(size=STYLE['legend_size']),
        yanchor='bottom', y=0.02, xanchor='right', x=0.98,
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#e5e7eb', borderwidth=1,
    ),
))

save_chart(fig_loco1,
           os.path.join(OUT, 'LOCO_CoefStability_model2_resource_rich'),
           width=1100, height=600)

# ── CHART 2: Sign consistency ─────────────────────────────────────────────────
plot_loco2 = loco_df.sort_values('Sign_Consist', ascending=True).reset_index(drop=True)

sign_colors = [
    PALETTE['green']  if v >= 0.90 else
    PALETTE['orange'] if v >= 0.70 else
    PALETTE['red']
    for v in plot_loco2['Sign_Consist']
]

fig_loco2 = go.Figure()

fig_loco2.add_trace(go.Bar(
    y=plot_loco2['Short'],
    x=plot_loco2['Sign_Consist'],
    orientation='h',
    marker=dict(color=sign_colors, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.0%}' for v in plot_loco2['Sign_Consist']],
    textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
    hovertemplate='<b>%{y}</b><br>Sign Consistency: %{x:.1%}<extra></extra>',
    showlegend=False,
))

for xval, color, dash, label in [
    (0.90, PALETTE['green'],  'dash', '90% consistent'),
    (0.70, PALETTE['orange'], 'dot',  '70% consistent'),
]:
    fig_loco2.add_vline(
        x=xval,
        line=dict(color=color, width=1.5, dash=dash),
        annotation_text=label,
        annotation_position='top',
        annotation_font=dict(size=9, color=color),
    )

# Dummy legend
for label, color in [
    ('High (≥90%)',     PALETTE['green']),
    ('Moderate (70–90%)', PALETTE['orange']),
    ('Low (<70%)',      PALETTE['red']),
]:
    fig_loco2.add_trace(go.Bar(
        y=[None], x=[None], orientation='h',
        marker=dict(color=color, line=dict(color='#1a2744', width=0.5)),
        name=label, showlegend=True,
    ))

fig_loco2.update_layout(**base_layout(
    height=STYLE['chart_height_tall'],
    margin=STYLE['margin_bar'],
    xaxis=dict(
        title=dict(text='Sign Consistency (proportion of LOCO runs)',
                   font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        range=[0, 1.18], tickformat='.0%',
    ),
    yaxis=dict(tickfont=dict(size=STYLE['tick_size'])),
    legend=dict(
        orientation='h', yanchor='top', y=-0.12,
        xanchor='center', x=0.5,
        font=dict(size=STYLE['legend_size']),
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#e5e7eb', borderwidth=1,
    ),
))

save_chart(fig_loco2,
           os.path.join(OUT, 'LOCO_SignConsistency_model2_resource_rich'),
           width=1100, height=700)

# ── CHART 3: R² stability across LOCO runs ────────────────────────────────────
r2_df = pd.DataFrame({
    'Country': loco_countries,
    'R2':      loco_r2,
}).sort_values('R2', ascending=True).reset_index(drop=True)

full_r2 = elastic.score(X_train, y_train)

r2_colors = [
    PALETTE['red']    if abs(v - full_r2) > 0.05 else
    PALETTE['orange'] if abs(v - full_r2) > 0.02 else
    PALETTE['green']
    for v in r2_df['R2']
]

fig_loco3 = go.Figure()

fig_loco3.add_trace(go.Bar(
    y=r2_df['Country'],
    x=r2_df['R2'],
    orientation='h',
    marker=dict(color=r2_colors, line=dict(color='#1a2744', width=0.5)),
    text=[f'{v:.3f}' for v in r2_df['R2']],
    textposition='outside',
    textfont=dict(size=STYLE['annotation_size'], color=STYLE['title_color']),
    hovertemplate='<b>%{y}</b><br>R²: %{x:.4f}<extra></extra>',
    showlegend=False,
))

# Full model R² reference line
fig_loco3.add_vline(
    x=full_r2,
    line=dict(color=PALETTE['blue'], width=2, dash='dash'),
    annotation_text=f'Full-sample R² = {full_r2:.3f}',
    annotation_position='top right',
    annotation_font=dict(size=10, color=PALETTE['blue']),
)

# Dummy legend
for label, color in [
    ('Stable (Δ < 0.02)',       PALETTE['green']),
    ('Moderate (0.02 ≤ Δ < 0.05)', PALETTE['orange']),
    ('Influential (Δ ≥ 0.05)', PALETTE['red']),
]:
    fig_loco3.add_trace(go.Bar(
        y=[None], x=[None], orientation='h',
        marker=dict(color=color, line=dict(color='#1a2744', width=0.5)),
        name=label, showlegend=True,
    ))

fig_loco3.update_layout(**base_layout(
    height=STYLE['chart_height_tall'],
    margin=STYLE['margin_bar'],
    xaxis=dict(
        title=dict(text='In-sample R² (leaving one country out)',
                   font=dict(size=STYLE['axis_title_size'])),
        gridcolor=STYLE['grid_color'], gridwidth=STYLE['grid_width'],
        range=[max(0, min(r2_df['R2']) * 0.95),
               max(r2_df['R2'].max(), full_r2) * 1.10],
    ),
    yaxis=dict(tickfont=dict(size=9)),
    legend=dict(
        orientation='h', yanchor='top', y=-0.12,
        xanchor='center', x=0.5,
        font=dict(size=STYLE['legend_size']),
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#e5e7eb', borderwidth=1,
    ),
))

save_chart(fig_loco3,
           os.path.join(OUT, 'LOCO_R2Stability_model2_resource_rich'),
           width=1100, height=700)

print(f"\n✓ LOCO analysis complete — 3 charts saved to {OUT}/")


── LOCO Robustness Check ───────────────────────────────────────────────
  LOCO runs completed: 56 countries

── LOCO Summary (sorted by |Full Coef|) ────────────────────────────────
               Short  Full_Coef  LOCO_Mean  LOCO_Std  LOCO_Min  LOCO_Max  Sign_Consist
     Domestic Credit     0.0244     0.0237    0.0016    0.0182    0.0279         1.000
         Electricity     0.0139     0.0097    0.0045    0.0019    0.0208         1.000
     HC × Production     0.0111     0.0084    0.0027    0.0029    0.0133         1.000
 Political Stability     0.0073     0.0056    0.0020    0.0000    0.0105         0.982
    Production Value    -0.0064    -0.0026    0.0039   -0.0114   -0.0000         0.321
         Rule of Law     0.0048     0.0039    0.0011    0.0000    0.0066         0.982
           Inflation    -0.0044    -0.0024    0.0017   -0.0055   -0.0000         0.982
        Depreciation    -0.0044    -0.0020    0.0024   -0.0073   -0.0000         0.804
               Trade    -0.0016  

  ✓ ..\figures\ml\LOCO_CoefStability_model2_resource_rich.html
  ✓ ..\figures\ml\LOCO_CoefStability_model2_resource_rich.png (kaleido)


  ✓ ..\figures\ml\LOCO_SignConsistency_model2_resource_rich.html
  ✓ ..\figures\ml\LOCO_SignConsistency_model2_resource_rich.png (kaleido)


  ✓ ..\figures\ml\LOCO_R2Stability_model2_resource_rich.html
  ✓ ..\figures\ml\LOCO_R2Stability_model2_resource_rich.png (kaleido)

✓ LOCO analysis complete — 3 charts saved to ..\figures\ml/
